# C3 · Dynamics: RMS energy and low-energy rate

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juansgomez87/py-mir-toolbox/blob/main/docs/chapters/C3_dynamics.ipynb)

Dynamics is how loud the music is and how that changes over time. MIRtoolbox measures it with the **root-mean-square (RMS) energy** of the waveform, either over the whole file or frame by frame, and summarises the RMS curve with the **low-energy rate**.

RMS is a physical measure of the signal's amplitude. It follows perceived loudness only roughly: it ignores the ear's frequency sensitivity, so a 50 Hz tone and a 3 kHz tone with the same RMS do not sound equally loud.

Each section below follows the course format: ① the musical idea, with sound; ② the maths from scratch in NumPy; ③ mirpy (MIRtoolbox semantics); ④ librosa; ⑤ Essentia; ⑥ why the numbers differ.

## Setup

In Colab, the next cell installs the libraries used in this chapter. On your own computer, see *Getting started*.

In [ ]:
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic(
        "pip", "install -q git+https://github.com/juansgomez87/py-mir-toolbox librosa essentia"
    )

In [ ]:
import librosa
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio

import mirpy as mir

try:
    import essentia.standard as es
except ImportError:
    es = None
    print("Essentia is not installed here; the Essentia cells will be skipped.")

## Framing in brief

Dynamics change over time, so most features are computed on short **frames**: windows of a few tens of milliseconds that slide along the signal. MIRtoolbox's default for RMS is a 50 ms frame with a hop of half a frame (25 ms).

Two details matter when comparing libraries:

- MIRtoolbox rounds the frame length *down* to whole samples, but keeps the hop as a fraction. At 22 050 Hz a 50 ms frame is 1102 samples (1102.5 rounded down) and the hop is 551.25 samples, so frame $i$ starts at sample $\lfloor 551.25\, i \rfloor$.
- Only whole frames are kept: the end of the signal that does not fill a frame is dropped.

The number of frames is therefore $\lfloor (N - L) / h \rfloor + 1$ for $N$ samples, frame length $L$ and hop $h$.

In [ ]:
sr = 22050
t = np.arange(2 * sr) / sr  # two seconds
tone = 0.5 * np.sin(2 * np.pi * 220 * t)

frame_seconds = 0.05
hop_ratio = 0.5
frame_length = int(frame_seconds * sr)  # 1102 samples
hop = hop_ratio * frame_seconds * sr  # 551.25 samples
n_frames = int((tone.size - frame_length) // hop) + 1
starts = np.floor(np.arange(n_frames) * hop).astype(int)
print(frame_length, hop, n_frames, starts[:5])

`mir.frame` does the same. The `frame` argument of every mirpy feature takes `(length in seconds, hop as a ratio of the length)`, like MATLAB `'Frame', 0.05, 0.5`. The framed signal has shape `(channels, samples per frame, frames)`.

In [ ]:
a = mir.audio(tone, sr=sr)
framed = mir.frame(a, (frame_seconds, hop_ratio))
print(framed.data.shape)
print(np.allclose(framed.frame_starts * sr, starts))

## RMS energy

### ① Idea

A sound is louder when its waveform swings further from zero. RMS squares every sample (so negative and positive swings both count), averages the squares, and takes the square root to return to the units of the waveform:

$$\mathrm{RMS} = \sqrt{\frac{1}{N} \sum_{n=1}^{N} x_n^2}$$

For a sine wave of amplitude $A$ this gives $A / \sqrt{2}$. Listen to a tone that grows from almost silent to loud. (The players below are not normalised, so the volume you hear follows the waveform.)

In [ ]:
envelope = np.linspace(0.01, 1, t.size)
crescendo = envelope * np.sin(2 * np.pi * 220 * t)
Audio(crescendo, rate=sr, normalize=False)

### ② From scratch

The RMS of the whole signal, then of every frame, using the frame starts computed above.

In [ ]:
def rms_from_scratch(x):
    """RMS of a whole signal: square, average, square root."""
    return np.sqrt(np.mean(x**2))


frames = crescendo[starts[:, np.newaxis] + np.arange(frame_length)]  # (frames, samples)
rms_curve = np.sqrt(np.mean(frames**2, axis=1))

print(f"whole signal: {rms_from_scratch(crescendo):.5f}")
print(f"first and last frame: {rms_curve[0]:.5f}, {rms_curve[-1]:.5f}")

### ③ mirpy

`mir.rms` is MIRtoolbox `mirrms`. Without `frame` it returns one value; with `frame=True` it uses the 50 ms / half-overlap default.

In [ ]:
c = mir.audio(crescendo, sr=sr)
whole = mir.rms(c)
curve = mir.rms(c, frame=True)

print(f"whole signal: {whole.data.item():.5f}")
print("frames match the NumPy version:", np.allclose(curve.data[0, 0], rms_curve))

The framed values match the NumPy version exactly, but the whole-signal value is slightly larger. This is a MIRtoolbox detail that mirpy keeps on purpose: for an **unframed** signal MIRtoolbox divides the sum of squares by $N - 1$, not $N$. For a framed signal it divides by $N$. We found this by comparing with MIRtoolbox 1.8.2 outputs (the manual's RMS of `ragtime.wav`, 0.01793, needs the $N - 1$). With two seconds of audio the difference is in the fifth decimal:

In [ ]:
n = crescendo.size
print(f"divide by N:     {np.sqrt(np.sum(crescendo**2) / n):.7f}")
print(f"divide by N - 1: {np.sqrt(np.sum(crescendo**2) / (n - 1)):.7f}")
print(f"mir.rms:         {whole.data.item():.7f}")

The RMS curve of the crescendo. The amplitude envelope grows linearly, and so does the RMS, at $1/\sqrt{2}$ of the envelope.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(curve.frame_centers, curve.data[0, 0], label="mir.rms (frames)")
ax.plot(t, envelope / np.sqrt(2), "--", label="envelope / √2")
ax.set_xlabel("Time (s)")
ax.set_ylabel("RMS")
ax.legend()
plt.show()

### ④ librosa

`librosa.feature.rms` takes the frame length and hop in samples. The hop must be a whole number, so 551.25 becomes 551. By default librosa also *centres* the frames: it pads the signal by half a frame at each end, so the first frame is centred on time 0.

In [ ]:
librosa_curve = librosa.feature.rms(
    y=crescendo, frame_length=frame_length, hop_length=int(hop), center=False
)[0]
print(librosa_curve.shape, curve.n_frames)
print(f"largest difference from mirpy: {np.max(np.abs(librosa_curve - curve.data[0, 0])):.2e}")

centred = librosa.feature.rms(y=crescendo, frame_length=frame_length, hop_length=int(hop))[0]
print(f"centred (librosa default): {centred.size} frames, first value {centred[0]:.5f}")

### ⑤ Essentia

Essentia separates framing (`FrameGenerator`) from the feature (`RMS`). It works in 32-bit floats. With `startFromZero=True` the first frame starts at sample 0, as in MIRtoolbox.

In [ ]:
def essentia_rms_curve(x):
    """RMS of each frame with Essentia, frames starting at sample 0."""
    frames = es.FrameGenerator(
        x.astype(np.float32), frameSize=frame_length, hopSize=int(hop), startFromZero=True
    )
    rms_algorithm = es.RMS()
    return np.array([rms_algorithm(frame) for frame in frames])


if es is None:
    print("Essentia is not available; skipped.")
else:
    essentia_curve = essentia_rms_curve(crescendo)
    print(essentia_curve.size, curve.n_frames)
    print(f"last frame: Essentia {essentia_curve[-1]:.5f}, mirpy {curve.data[0, 0, -1]:.5f}")

### ⑥ Why the numbers differ

All four versions compute the same formula. The differences come from how the signal is cut into frames and how the result is stored:

| | NumPy above | mirpy (MIRtoolbox) | librosa | Essentia |
|---|---|---|---|---|
| Hop | fractional | fractional (551.25) | whole samples | whole samples |
| First frame | starts at 0 | starts at 0 | centred on 0 (default) | starts at 0 with `startFromZero=True` |
| End of signal | dropped | dropped | dropped (`center=False`) | last frame zero-padded and kept |
| Whole-signal divisor | $N$ | $N - 1$ | (frames only) | $N$ |
| Precision | float64 | float64 | input dtype (float32 from `librosa.load`) | float32 |

The integer hop shifts librosa's and Essentia's frames by up to a few samples against MIRtoolbox's, so their curves differ in about the fourth decimal. The extra zero-padded frame at the end gives Essentia one more value, smaller than its neighbours because part of it is silence. For statistics on whole songs these differences are negligible. To reproduce MIRtoolbox results exactly, use mirpy. To combine RMS with other librosa features, use librosa's frames so that all features share the same time axis.

## Low-energy rate

### ① Idea

The low-energy rate is the share of frames whose RMS is below the average RMS of the whole excerpt (Tzanetakis & Cook, 2002). Music with a steady level gives a value around 0.5. Music made of a few loud events separated by quiet passages gives a high value: the loud events raise the average, and most frames fall below it.

Two synthetic examples: a sustained tone with a slow tremolo, and a series of short percussive hits.

In [ ]:
sustained = (0.5 + 0.1 * np.sin(2 * np.pi * 2 * t)) * np.sin(2 * np.pi * 220 * t)

rng = np.random.default_rng(0)
hit_times = np.arange(0, 2, 0.25)
decay = np.exp(-(t[:, np.newaxis] - hit_times) / 0.02) * (t[:, np.newaxis] >= hit_times)
percussive = decay.sum(axis=1) * rng.standard_normal(t.size)
percussive *= 0.9 / np.max(np.abs(percussive))  # peak at 0.9, no clipping

Audio(sustained, rate=sr, normalize=False)

In [ ]:
Audio(percussive, rate=sr, normalize=False)

### ② From scratch

In [ ]:
def framed_rms(x):
    """RMS of each frame, with the frame starts computed above."""
    frames = x[starts[:, np.newaxis] + np.arange(frame_length)]
    return np.sqrt(np.mean(frames**2, axis=1))


def low_energy_rate(x):
    """Share of frames whose RMS is below the average RMS."""
    energy = framed_rms(x)
    return np.mean(energy < energy.mean())


print(f"sustained:  {low_energy_rate(sustained):.3f}")
print(f"percussive: {low_energy_rate(percussive):.3f}")

### ③ mirpy

`mir.lowenergy` is MIRtoolbox `mirlowenergy`. It takes the waveform (and frames it with the RMS default) or an RMS curve you computed yourself.

In [ ]:
for name, x in [("sustained", sustained), ("percussive", percussive)]:
    value = mir.lowenergy(mir.audio(x, sr=sr)).data.item()
    print(f"{name}: {value:.3f}")

Two options change the definition:

- `threshold`: a frame counts as low when its energy is below `threshold` times the average (MATLAB `'Threshold'`, default 1).
- `asr=True`: the **Average Silence Ratio** (Feng, Zhuang & Pan, 2003) uses the mean square (RMS without the square root) and a threshold of 0.5 (MATLAB `'ASR'`).

In [ ]:
print("            default  threshold 0.9  ASR")
for name, x in [("sustained", sustained), ("percussive", percussive)]:
    signal = mir.audio(x, sr=sr)
    values = [
        mir.lowenergy(signal).data.item(),
        mir.lowenergy(signal, threshold=0.9).data.item(),
        mir.lowenergy(signal, asr=True).data.item(),
    ]
    print(f"{name:<12}" + "".join(f"{v:<15.3f}" for v in values))

Lowering the threshold to 0.9 reduces the rate of the sustained tone, whose frames all sit near the average, but does not change the percussive one, whose quiet frames are far below it. For the same reason the ASR of the sustained tone is 0: no frame has less than half the average energy.

### ④ librosa and ⑤ Essentia

Neither library has a low-energy function, but it is one line on top of their RMS curves. The result matches mirpy closely, apart from the framing differences shown above.

In [ ]:
librosa_energy = librosa.feature.rms(
    y=percussive, frame_length=frame_length, hop_length=int(hop), center=False
)[0]
print(f"librosa: {np.mean(librosa_energy < librosa_energy.mean()):.3f}")

if es is not None:
    essentia_energy = essentia_rms_curve(percussive)
    print(f"Essentia: {np.mean(essentia_energy < essentia_energy.mean()):.3f}")

### ⑥ Why the numbers differ

The low-energy rate counts frames, so it depends on how many frames there are and where they fall. Shifting the frames by a sample or adding Essentia's padded last frame can move a frame from one side of the average to the other, which changes the rate by $1/\text{(number of frames)}$. With 79 frames that is about 0.013; on a three-minute song it is below 0.001.

The rate also depends on the frame length: with longer frames, short hits are averaged with the silence around them and the rate drops. Report the frame settings with the value.

## A real recording

An excerpt of Brahms's Hungarian Dance No. 5, played by the US Army Strings (public domain, via [Musopen](https://musopen.org/music/43805-hungarian-dance-no-5-in-f-sharp-minor-woo-1-string-orch); downloaded by librosa the first time it runs). The piece alternates loud *tutti* passages with softer, slower ones.

In [ ]:
brahms = mir.audio(librosa.ex("brahms"))
Audio(brahms.data[0, :, 0], rate=brahms.sr)

In [ ]:
brahms_rms = mir.rms(brahms, frame=True)

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(brahms_rms.frame_centers, brahms_rms.data[0, 0], linewidth=0.8)
ax.axhline(brahms_rms.data.mean(), color="gray", linestyle="--", label="average RMS")
ax.set_xlabel("Time (s)")
ax.set_ylabel("RMS")
ax.legend()
plt.show()

print(f"RMS of the whole excerpt: {mir.rms(brahms).data.item():.4f}")
print(f"low-energy rate:          {mir.lowenergy(brahms_rms).data.item():.3f}")

The low-energy rate is close to 0.5, as for the sustained tone: the strings keep a fairly even level between the loud and soft passages, so the RMS curve spends about as much time below its average as above it.

## Exercise: RMS in decibels

The crescendo above grows linearly in amplitude, but we hear loudness roughly on a logarithmic scale. Convert the RMS curve of the crescendo to decibels, $20 \log_{10}(\mathrm{RMS})$, and plot it. Where in the crescendo does the level change fastest, in dB per second? Does that match what you hear?

Then build a crescendo that grows linearly *in decibels* (from −40 dB to 0 dB) and plot its RMS in dB.

In [ ]:
# Your answer here.

A solution (click to show):

In [ ]:
rms_db = 20 * np.log10(curve.data[0, 0])

db_envelope = 10 ** (np.linspace(-40, 0, t.size) / 20)
db_crescendo = db_envelope * np.sin(2 * np.pi * 220 * t)
db_curve = mir.rms(mir.audio(db_crescendo, sr=sr), frame=True)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(curve.frame_centers, rms_db, label="linear-amplitude crescendo")
ax.plot(db_curve.frame_centers, 20 * np.log10(db_curve.data[0, 0]), label="linear-dB crescendo")
ax.set_xlabel("Time (s)")
ax.set_ylabel("RMS (dB)")
ax.legend()
plt.show()

The linear-amplitude crescendo rises fastest in dB at the start, then flattens: most of its change in level happens in the first half-second. The linear-dB crescendo rises at a constant 20 dB per second, which sounds more even.

## Summary

| MIRtoolbox | mirpy | librosa | Essentia |
|---|---|---|---|
| `mirrms(a)` | `mir.rms(a)` | `np.sqrt(np.mean(y**2))` | `es.RMS()(y)` |
| `mirrms(a, 'Frame')` | `mir.rms(a, frame=True)` | `librosa.feature.rms(y=y, center=False)` | `FrameGenerator` + `RMS` |
| `mirlowenergy(a)` | `mir.lowenergy(a)` | from `feature.rms` | from `RMS` |

## Next

C4 Timbre describes the spectrum of each frame: brightness, spectral shape, MFCCs and spectral flux.

## References

- Feng, Y., Zhuang, Y., & Pan, Y. (2003). Popular music retrieval by detecting mood. *Proceedings of ACM SIGIR*, 375–376.
- Lartillot, O., & Toiviainen, P. (2007). A Matlab toolbox for musical feature extraction from audio. *Proceedings of DAFx*.
- Tzanetakis, G., & Cook, P. (2002). Musical genre classification of audio signals. *IEEE Transactions on Speech and Audio Processing*, 10(5), 293–302.